In [ ]:
import pandas as pd

raw_prices = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\raw_prices.csv", index_col=0, parse_dates=True)
adj_prices = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\adjusted_prices.csv", index_col=0, parse_dates=True)
raw_volume = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\raw_volume.csv", index_col=0, parse_dates=True)

adj_prices = pd.DataFrame(adj_prices)

NameError: name 'pd' is not defined

In [ ]:
print(raw_prices.head())


In [ ]:
print(adj_prices.head())

In [ ]:
adj_prices.info()
adj_prices.shape
adj_prices.isna().sum()


In [ ]:
adj_prices.duplicated().sum()

In [ ]:
for column in adj_prices.columns:
    for index in adj_prices.index:
        if adj_prices.loc[index,column]<=0:
            print(f"Error: Negative or zero value found in column '{column}' at index '{index}' with value {adj_prices.loc[index,column]}")

In [ ]:
adj_returns = adj_prices.pct_change()  
print(adj_returns.head())

for column in adj_returns.columns:
    for index in adj_returns.index:
        if adj_returns.loc[index,column]<=-0.5:
            print(f"Error: Return less than or equal to -100% found in column '{column}' at index '{index}' with value {adj_returns.loc[index,column]}")
        elif adj_returns.loc[index,column]>=0.5:
            print(f"Error: Return greater than or equal to 50% found in column '{column}' at index '{index}' with value {adj_returns.loc[index,column]}")

In [ ]:
adj_prices.describe()

In [ ]:
adj_returns.describe()

In [ ]:
adj_prices_m = adj_prices.resample('ME').last()
returns_m = adj_prices_m.pct_change(fill_method = None)
print(adj_prices_m.shape, returns_m.shape)
print(adj_prices_m.head())
volume_m = raw_volume.resample('ME').sum()
raw_prices_m = raw_prices.resample('ME').last()
dollar_volume_m = raw_prices_m * volume_m

In [ ]:
price_filter = raw_prices_m >= 5
volume_filter = dollar_volume_m > 1_000_000
return_filter = returns_m.notna()

fail_price  = (~price_filter).sum().sort_values(ascending=False)
fail_volume = (~volume_filter).sum().sort_values(ascending=False)
fail_return = (~return_filter).sum().sort_values(ascending=False)

summary = pd.DataFrame({
    "fail_price": fail_price,
    "fail_volume": fail_volume,
    "fail_return": fail_return,
    "fail_any": (~eligible).sum()
}).sort_values("fail_any", ascending=False)

print(summary)

In [ ]:
eligible = price_filter & volume_filter & return_filter

def pct_excluded(mask, label):
    total = mask.size
    excluded = (~mask).sum().sum()
    pct = excluded / total * 100
    print(f"{label}: {excluded:,} / {total:,} stock-months excluded ({pct:.1f}%)")
    return pct

pct_excluded(price_filter,  "Price filter (<$5)")
pct_excluded(volume_filter, "Volume filter (<$1m dollar volume)")
pct_excluded(return_filter, "Missing/invalid return")
pct_excluded(eligible,      "Combined (any filter)")

In [ ]:
adj_returns.to_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\adj_returns.csv")
returns_m.to_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\returns_m.csv")
volume_m.to_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\volume_m.csv")
raw_prices_m.to_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\raw_prices_m.csv")